In [ ]:
pip
install
tree_sitter
tree_sitter_languages
tree - sitter - go

In [1]:
from __future__ import annotations  # ← оставляем самым первым

import os
import tempfile
from typing import Dict

# глушим лишние треды от нативных либ до их импорта
os.environ.setdefault("OMP_NUM_THREADS", "1")

from tree_sitter import Parser, Language
import tree_sitter_go as tsgo  # используем tsgo.language()

In [2]:
# Если не включён в ноутбуке ранее:
# from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Tuple


# ──────────────────────────────────────────────────────────────────────────────
# Go models
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class GoImport:
    path: str
    alias: Optional[str] = None


@dataclass
class GoField:
    """Поле структуры Go. Если type == 'struct { ... }', то fields содержит вложенные поля."""
    name: str  # Имя поля (или тип для embedded)
    type: str  # Текст типа как в коде (вкл. inline: "struct {...}")
    tag: Optional[str] = None  # Тег (без кавычек), если есть
    embedded: bool = False  # Встраиваемое поле (embedded)
    fields: Optional[List["GoField"]] = None  # Для inline-struct; None если не struct


@dataclass
class GoType:
    """Тип верхнего уровня: struct/interface/alias/other."""
    name: str
    kind: str  # struct|interface|alias|other
    fields: Optional[List[Tuple[str, str]]] = None  # [(name, type)] для struct/alias
    methods: Optional[List[str]] = None  # интерфейсы: сигнатуры; alias: [target]
    line: Optional[int] = None


@dataclass
class GoParam:
    """Параметр функции/метода Go."""
    name: Optional[str]  # может отсутствовать (анонимный)
    type: str  # int, *T, pkg.X, []byte, map[string]int, chan T, ...
    variadic: bool = False  # ...T


@dataclass
class GoFunc:
    """Функция или метод Go."""
    name: str
    receiver: Optional[str] = None  # "T" или "*T" для методов
    exported: bool = False
    params: List[GoParam] = field(default_factory=list)  # входные параметры
    results: List[str] = field(default_factory=list)  # выходные типы
    doc: Optional[str] = None
    line: Optional[int] = None


@dataclass
class GoFileMeta:
    """Метаданные по одному .go файлу."""
    path: str
    package: Optional[str]
    imports: List[GoImport] = field(default_factory=list)
    types: List[GoType] = field(default_factory=list)
    funcs: List[GoFunc] = field(default_factory=list)


# ──────────────────────────────────────────────────────────────────────────────
# Proto models
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class ProtoField:
    """Поле message в .proto."""
    name: str
    type: str
    number: int
    label: Optional[str] = None  # optional|repeated (proto3), required (proto2)
    default: Optional[str] = None  # текстовое значение по умолчанию (если указано)


@dataclass
class ProtoMessage:
    """Message в .proto."""
    name: str
    fields: List[ProtoField] = field(default_factory=list)


@dataclass
class ProtoEnumValue:
    name: str
    number: int


@dataclass
class ProtoEnum:
    """Enum в .proto."""
    name: str
    values: List[ProtoEnumValue] = field(default_factory=list)


@dataclass
class ProtoServiceMethod:
    """RPC-метод сервиса."""
    name: str
    input_type: str  # имя message запроса
    output_type: str  # имя message ответа
    client_streaming: bool = False
    server_streaming: bool = False


@dataclass
class ProtoService:
    """Service в .proto."""
    name: str
    methods: List[ProtoServiceMethod] = field(default_factory=list)


@dataclass
class ProtoFileMeta:
    """Метаданные по одному .proto файлу."""
    path: str
    package: Optional[str] = None
    messages: List[ProtoMessage] = field(default_factory=list)
    enums: List[ProtoEnum] = field(default_factory=list)
    services: List[ProtoService] = field(default_factory=list)

In [3]:
# ──────────────────────────────────────────────────────────────────────────────
# Инициализация tree-sitter-go
# ──────────────────────────────────────────────────────────────────────────────
def _load_go_language() -> Language:
    """
    Загружает Language для Go строго из tree-sitter-go.

    Способы:
      1) Среда: TREE_SITTER_GO_SO=/abs/path/to/go.so  (готовая сборка)
      2) Среда: TREE_SITTER_GO_GRAMMAR=/abs/path/to/tree-sitter-go  (автосборка)
         → соберём .so во временную дирекцию (нужен gcc/clang).

    Исключение — если ничего не найдено.
    """
    so_path = os.getenv("TREE_SITTER_GO_SO")
    if so_path:
        return Language(so_path, "go")

    grammar_dir = os.getenv("TREE_SITTER_GO_GRAMMAR")
    if grammar_dir:
        # Сборка .so на лету
        out_dir = Path(tempfile.gettempdir()) / "tsgo_build"
        out_dir.mkdir(parents=True, exist_ok=True)
        lib_path = str(out_dir / "tree_sitter_go.so")
        # Важно: Language.build_library перезапишет, если файл уже есть
        Language.build_library(
            # итоговая общая библиотека
            lib_path,
            # пути до репозиториев грамматик
            [grammar_dir],
        )
        return Language(lib_path, "go")

    raise RuntimeError(
        "Не найден tree-sitter-go. "
        "Установите один из вариантов:\n"
        "  • export TREE_SITTER_GO_SO=/abs/path/to/tree_sitter_go.so\n"
        "  • export TREE_SITTER_GO_GRAMMAR=/abs/path/to/tree-sitter-go   # исходники грамматики\n"
        "Пример грамматики: https://github.com/tree-sitter/tree-sitter-go"
    )


def make_go_parser() -> Parser:
    p = Parser(Language(tsgo.language()))
    return p


In [4]:
# ──────────────────────────────────────────────────────────────────────────────
# Обход проекта
# ──────────────────────────────────────────────────────────────────────────────

from pathlib import Path
from typing import Optional, Set, Union, Iterable

GO_FILE_EXTS = frozenset({".go"})
GO_AUX_FILES = frozenset({"go.mod", "go.sum"})


def _is_ignored_dir(path: Path, ignore_dirs: Set[str]) -> bool:
    nm = path.name
    return (
            nm in ignore_dirs
            or (nm.startswith(".") and nm not in {".", ".."})
            or nm in {"vendor", "node_modules", "dist", "build", "out"}
    )


def iter_go_files(PROJECT_PATH: Union[str, Path], IGNORE_DIRS: Iterable[str]) -> Iterable[Path]:
    """
    Ищет .go и служебные файлы (go.mod/go.sum), пропуская игнорируемые директории.
    Без изменения логики: остаёмся на rglob, но ускоряем проверку родителей.
    """
    root = Path(PROJECT_PATH).resolve()
    ignore = {d.strip("/").strip() for d in IGNORE_DIRS if d}

    for p in root.rglob("*"):
        if p.is_dir():
            continue
        # Быстрые отсеки до проверки родителей
        if p.suffix not in GO_FILE_EXTS and p.name not in GO_AUX_FILES:
            continue
        # Проверяем, не попадает ли любой сегмент пути в игнор
        # (эквивалентно твоей проверке родителей, но без обхода всех parent до root.parent)
        if any(_is_ignored_dir(Path(part), ignore) for part in p.parts):
            # Примечание: Path(part) здесь используется только для name,
            # можно заменить на простой объект с .name = part, но так чище.
            continue
        yield p


# ──────────────────────────────────────────────────────────────────────────────
# Парсинг одного файла
# ──────────────────────────────────────────────────────────────────────────────

def _text(src: bytes, node) -> str:
    # Унифицируем извлечение текста: один helper вместо двух
    return src[node.start_byte:node.end_byte].decode("utf-8", errors="ignore")


def _strip_quotes(s: str) -> str:
    s = s.strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in {'"', '`'}:
        return s[1:-1]
    return s


def _string_literal_text(src: bytes, node) -> Optional[str]:
    if node.type in ("interpreted_string_literal", "raw_string_literal"):
        return _strip_quotes(_text(src, node))
    return None


def _collect_leading_comments(src: bytes, node) -> Optional[str]:
    """
    Собирает подряд идущие сверху комментарии (// ... или /* ... */),
    непосредственно прилегающие к node, без пустых строк между ними.
    Текущая реализация оставлена простой: ищем среди комментов предков.
    """
    lines = []
    prev_end_row = node.start_point[0] - 1  # строка перед узлом
    parent = node.parent
    if parent is None:
        return None

    # Пробегаем только по комментариям под тем же родителем (не по всему файлу)
    for com in _all_descendants_of_type(parent, "comment"):
        if com.end_point[0] == prev_end_row:
            text = _text(src, com)
            lines.insert(0, text)  # вставляем в начало, чтобы сохранить порядок сверху-вниз
            # уменьшаем prev_end_row на число строк комментария
            # (для /* ... */ многострочных блоков)
            prev_end_row -= (text.count("\n") or 0)

    return "\n".join(lines) if lines else None


# ──────────────────────────────────────────────────────────────────────────────
# AST helpers — без Query
# ──────────────────────────────────────────────────────────────────────────────

def _children(node):
    # Простой генератор по дочерним
    for ch in node.children:
        yield ch


def _first_child_of_type(node, t: str):
    for ch in node.children:
        if ch.type == t:
            return ch
    return None


def _all_descendants_of_type(node, t: str):
    stack = [node]
    while stack:
        n = stack.pop()
        if n.type == t:
            yield n
        # extend без создания лишних списков
        if n.children:
            stack.extend(n.children)

In [5]:
# ──────────────────────────────────────────────────────────────────────────────
# Парсер типов и вложенных структур (+ хелперы для proto)
# ──────────────────────────────────────────────────────────────────────────────

def _extract_field_tag(src: bytes, decl_node) -> Optional[str]:
    """
    Возвращает строку тега поля (без кавычек) если в декларации поля есть завершающий string literal.
    В tree-sitter-go теги представлены как raw|interpreted string literal в конце декларации.
    """
    tag: Optional[str] = None
    for ch in _children(decl_node):
        if ch.type in ("interpreted_string_literal", "raw_string_literal"):
            tag = _strip_quotes(_text(src, ch))
    return tag


def _first_type_node_in_field_decl(decl_node):
    """
    Возвращает первый узел типа в field_declaration:
    (struct_type | pointer_type | slice_type | qualified_type | array_type | map_type | type_identifier | ...)
    """
    for ch in _children(decl_node):
        if ch.type in (
                "struct_type", "pointer_type", "slice_type", "qualified_type",
                "array_type", "map_type", "channel_type", "generic_type",
                "type_identifier", "union_type", "function_type",
        ):
            return ch
    return None


# ──────────────────────────────────────────────────────────────────────────────
# Хелперы распаковки типов до базового идентификатора (для proto matching)
# ──────────────────────────────────────────────────────────────────────────────

def unwrap_type_ident(src: bytes, type_node) -> Tuple[Optional[str], Optional[str]]:
    """
    Разворачивает тип до базового идентификатора и pkg-алиаса.
    Возвращает кортеж (pkg_alias, type_name), где pkg_alias может быть None.

    Поддержка:
      * pointer_type: *T
      * slice_type/array_type: []T / [N]T
      * channel_type: chan T
      * qualified_type: pkg.T
      * type_identifier: T
      * generic_type: Foo[Bar] -> берём Foo как идентификатор
    """
    if type_node is None:
        return None, None

    n = type_node
    # снимаем слои-обёртки
    while n.type in ("pointer_type", "slice_type", "array_type", "channel_type", "parenthesized_type"):
        # у всех этих узлов тип лежит в единственном child-е типа
        ch = _first_type_node_in_field_decl(n) or (n.children[0] if n.children else None)
        if ch is None:
            return None, None
        n = ch

    if n.type == "qualified_type":
        # qualified_type обычно: package_identifier "." type_identifier
        pkg_ident = _first_child_of_type(n, "package_identifier")
        typ_ident = _first_child_of_type(n, "type_identifier")
        pkg_alias = _text(src, pkg_ident) if pkg_ident else None
        type_name = _text(src, typ_ident) if typ_ident else None
        return pkg_alias, type_name

    if n.type == "generic_type":
        # generic_type: идентификатор с параметрами (Go 1.18+). Берём базовое имя типа.
        base = _first_child_of_type(n, "type_identifier") or _first_child_of_type(n, "qualified_type")
        if base is None:
            return None, None
        return unwrap_type_ident(src, base)

    if n.type == "type_identifier":
        return None, _text(src, n)

    # В остальных случаях (map_type, function_type, struct_type, union_type)
    # это не proto message имя.
    return None, None


def _is_json_str(x: Any) -> bool:
    return isinstance(x, str) and (x.lstrip().startswith("{") or x.lstrip().startswith("["))

def _get(obj: Any, attr: str, default=None):
    if isinstance(obj, dict):
        return obj.get(attr, default)
    return getattr(obj, attr, default)

def _iter(obj: Any, attr: str) -> List[Any]:
    v = _get(obj, attr, None)
    return list(v) if v else []

def _sanitize_ident(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z_]', '_', s)

def _index_structs_and_aliases(structs_map: Dict[str, Any]):
    """
    Возвращает:
      struct_names: Set[str]
      alias_rhs: Dict[alias -> rhs_text]
      items: List[(rel_path, pkg, type_obj)] только для struct
      name_counts: Dict[type_name -> count]
    """
    struct_names: Set[str] = set()
    alias_rhs: Dict[str, str] = {}
    items: List[Tuple[str, str, Any]] = []
    name_counts: Dict[str, int] = {}

    for rel_path, meta in structs_map.items():
        pkg = _get(meta, "package", "") or ""
        for t in _iter(meta, "types"):
            kind = _get(t, "kind")
            name = _get(t, "name")
            if not name:
                continue
            if kind == "struct":
                items.append((rel_path, pkg, t))
                struct_names.add(name)
                name_counts[name] = name_counts.get(name, 0) + 1
            elif kind == "alias":
                rhs_list = _iter(t, "methods")  # у нас RHS alias хранится тут
                rhs = rhs_list[0] if rhs_list else None
                if rhs:
                    alias_rhs[name] = str(rhs)
    return struct_names, alias_rhs, items, name_counts

def detect_proto_ref(
        pkg_alias: Optional[str],
        type_name: Optional[str],
        proto_index: Optional[Dict[str, object]] = None,
        known_proto_aliases: Optional[Set[str]] = None,
):
    """
    Пытается сопоставить (pkg_alias, type_name) с proto message через индекс.
    Ожидаемый формат proto_index ключей:
      - "Alias.MessageName" (например, "pb.GetUserRequest")
      - "MessageName"       (на случай уникальных имён)
    Возвращает объект из индекса (например, ProtoMessage) или None.
    """
    if not proto_index or not type_name:
        return None

    # приоритет: alias-qualified
    if pkg_alias:
        key = f"{pkg_alias}.{type_name}"
        if key in proto_index:
            return proto_index[key]

    # если алиас не задан, но это точно pb-алиас (в сигнатурах методов часто пишут "*pb.X")
    if known_proto_aliases and pkg_alias in known_proto_aliases:
        key = f"{pkg_alias}.{type_name}"
        if key in proto_index:
            return proto_index[key]

    # fallback — по голому имени (если уникально)
    return proto_index.get(type_name)


# ──────────────────────────────────────────────────────────────────────────────
# Разбор полей структуры
# ──────────────────────────────────────────────────────────────────────────────

def _parse_struct_fields(
        src: bytes,
        field_decl_list_node,
        embedded: bool = False,
        *,
        proto_index: Optional[Dict[str, object]] = None,
        known_proto_aliases: Optional[Set[str]] = None,
) -> List[GoField]:
    """
    Разбирает field_declaration_list внутри struct_type -> список GoField.
    Поддерживает:
      - несколько имён на один тип: A, B int
      - embedded поля: *Other / Other / pkg.Type
      - inline struct: X struct { ... }
      - теги у полей
    Опционально: proto_index/known_proto_aliases для последующего сопоставления
    (детект не записывается в GoField; если нужно — добавим поле в модель).
    """
    out: List[GoField] = []

    for decl in _all_descendants_of_type(field_decl_list_node, "field_declaration"):
        # Имена (могут быть отсутствующими для embedded)
        names = [_text(src, n) for n in _all_descendants_of_type(decl, "field_identifier")]
        embedded_node = _first_child_of_type(decl, "embedded_field")
        tag = _extract_field_tag(src, decl)
        type_node = _first_type_node_in_field_decl(decl)

        # Тип как текст (важно сохранить исходный вид, включая "struct {...}")
        type_text = _text(src, type_node).strip() if type_node else ""

        # Если это inline struct, рекурсивно распарсим вложенные поля
        subfields: Optional[List[GoField]] = None
        if type_node and type_node.type == "struct_type":
            fld_list = _first_child_of_type(type_node, "field_declaration_list")
            if fld_list:
                subfields = _parse_struct_fields(
                    src, fld_list, True,
                    proto_index=proto_index,
                    known_proto_aliases=known_proto_aliases,
                )

        # ── Proto detection (опционально, без побочных эффектов) ─────────────
        pkg_alias, base_type = unwrap_type_ident(src, type_node) if type_node else (None, None)
        _proto_msg = detect_proto_ref(pkg_alias, base_type, proto_index, known_proto_aliases)
        # Примечание: мы не сохраняем _proto_msg в GoField — следуем текущей модели.
        # Если потребуется — можно расширить GoField отдельным полем.

        if embedded_node and not names:
            # embedded поле — используем тип как имя
            emb_text = _text(src, embedded_node).strip()
            out.append(GoField(
                name=emb_text,
                type=type_text or emb_text,
                tag=tag,
                embedded=embedded,
                fields=subfields
            ))
            continue

        # Обычные именованные поля (возможны несколько имён под один тип)
        if names:
            for nm in names:
                out.append(GoField(
                    name=nm,
                    type=type_text,
                    tag=tag,
                    embedded=embedded,
                    fields=subfields
                ))
        else:
            # Теоретически: анонимное поле без embedded (редко), но поддержим
            out.append(GoField(
                name=type_text or "<anonymous>",
                type=type_text,
                tag=tag,
                embedded=embedded,
                fields=subfields
            ))

    return out

In [81]:
# ──────────────────────────────────────────────────────────────────────────────
# Парсер параметров в функциях (точечный тюнинг без смены логики)
# ──────────────────────────────────────────────────────────────────────────────

# Какие узлы считаем "шумом" при поиске типа
_SKIP_IN_TYPE = {"identifier", "comment", ",", "ellipsis"}  # ellipsis = '...'


def _node_sexpr(src: bytes, node) -> str:
    # Универсальный способ получить «текст типа» для узла
    return _text(src, node).strip()


def _first_type_child(node):
    """
    Возвращает первый дочерний узел, который выглядит как тип (а не имя/комментарий/запятая/ellipsis).
    Работает и для parameter_declaration, и для variadic_parameter_declaration.
    """
    for ch in node.children:
        if ch.type not in _SKIP_IN_TYPE:
            return ch
    return None


def _param_type_text(src: bytes, decl_node) -> Optional[str]:
    """
    В parameter_declaration тип — первый осмысленный дочерний узел (не имя/коммент/запятая).
    """
    tnode = _first_type_child(decl_node)
    if tnode is None:
        return None
    tt = _text(src, tnode).strip()
    return tt or None


def _parse_parameter_list(src: bytes, plist_node) -> List[GoParam]:
    """
    Разбирает (parameter_list) → [GoParam,...]
    Поддерживает:
      - parameter_declaration с несколькими идентификаторами, разделяющими один тип
      - variadic_parameter_declaration ( ...T / name ...T )
      - анонимные параметры (только тип, без имени)
    """
    params: List[GoParam] = []

    for ch in _children(plist_node):
        t = ch.type
        if t == "parameter_declaration":
            # Имена строго из прямых детей-типа identifier (не из qualified_type)
            direct_names = [_text(src, n) for n in ch.children if n.type == "identifier"]
            ptype = _param_type_text(src, ch) or ""
            if direct_names:
                for nm in direct_names:
                    params.append(GoParam(name=nm, type=ptype, variadic=False))
            else:
                # анонимный параметр
                params.append(GoParam(name=None, type=ptype, variadic=False))

        elif t == "variadic_parameter_declaration":
            # Вариадик: возможны формы "name ...T" и "...T"
            # Тип берём как первый осмысленный дочерний узел (после '...' и, возможно, имени)
            direct_name = None
            for c2 in ch.children:
                if c2.type == "identifier":
                    direct_name = _text(src, c2)
                    break
            tnode = _first_type_child(ch)
            ptype = _text(src, tnode).strip() if tnode else ""
            params.append(GoParam(name=direct_name, type=ptype, variadic=True))

        else:
            # пропускаем запятые/скобки и т.п.
            continue

    return params


# ──────────────────────────────────────────────────────────────────────────────
# Улучшенный парсинг результатов функции/метода в Go (для tree-sitter-go)
# ──────────────────────────────────────────────────────────────────────────────
def _parse_results(src: bytes, fn_or_sig_node) -> List[str]:
    """
    Возвращает список текстов типов результатов для func/method.
    Поддерживает:
      • (T1, T2) / (named T1, named2 T2) — берём типы из следующего parameter_list
      • одиночный тип без скобок: error, *pkg.Reply, []byte, map[K]V, chan T, ...
    Логика:
      1) Определяем индекс ВХОДНОГО списка параметров:
         - function_declaration: первый parameter_list
         - method_declaration: второй parameter_list (после ресивера), если есть; иначе первый.
      2) После него ищем:
         - следующий parameter_list → результаты (вытаскиваем типы)
         - либо одиночный узел-типа → один результат.
    """
    results: List[str] = []
    children = list(fn_or_sig_node.children)

    # Найдём все parameter_list
    pl_indices = [i for i, n in enumerate(children) if n.type == "parameter_list"]
    if not pl_indices:
        return results

    # Определим индекс списка ВХОДНЫХ параметров
    if fn_or_sig_node.type == "method_declaration":
        # [0] — ресивер, [1] — входы (если есть)
        in_pl_idx = pl_indices[1] if len(pl_indices) >= 2 else pl_indices[0]
    else:
        # function_declaration
        in_pl_idx = pl_indices[0]

    # Хвост после входного списка
    tail = children[in_pl_idx + 1:]
    if not tail:
        return results

    SKIP_TYPES = {
        "comment", ",",
        "field_identifier", "identifier",  # имя метода/функции и т.п.
        "type_parameters", "where_clause",
        "attribute_list", "attribute",
    }

    RESULT_TYPE_NODES = {
        "type_identifier", "qualified_type",
        "pointer_type", "slice_type", "array_type",
        "map_type", "channel_type", "generic_type",
        "function_type", "struct_type", "interface_type",
        "parenthesized_type",
    }

    out_pl_node = None
    single_type_node = None

    for n in tail:
        if n.type == "block":
            break
        if n.type in SKIP_TYPES:
            continue
        if n.type == "parameter_list":
            out_pl_node = n
            break
        if n.type in RESULT_TYPE_NODES:
            single_type_node = n
            break
        # Подстраховка: если дерево даёт уже собранный тип единым узлом
        if n.type not in (";",):
            single_type_node = n
            break

    # Вариант А: результаты в скобках — второй parameter_list
    if out_pl_node is not None:
        for ch in _children(out_pl_node):
            if ch.type in ("parameter_declaration", "variadic_parameter_declaration"):
                t = _param_type_text(src, ch)
                if t:
                    results.append(t.strip())
        return results

    # Вариант Б: одиночный тип
    if single_type_node is not None:
        txt = _node_sexpr(src, single_type_node).strip()
        if txt:
            results.append(txt)
        return results

    return results

In [82]:
# ──────────────────────────────────────────────────────────────────────────────
# Парсинг одного файла — только навигация по типам узлов (микро-оптимизация)
# ──────────────────────────────────────────────────────────────────────────────

# константы для проверок типов узлов (ускоряют membership)
_STR_LITS = ("interpreted_string_literal", "raw_string_literal")


def parse_go_file_structures(path: Path, parser: Parser) -> GoFileMeta:
    src = path.read_bytes()
    tree = parser.parse(src)
    root = tree.root_node

    # локальные ссылки (чуть быстрее, меньше глобальных lookup)
    _t = _text
    _fc = _first_child_of_type
    _ad = _all_descendants_of_type

    # ---- package ----
    package: Optional[str] = None
    pkg_clause = _fc(root, "package_clause")
    if pkg_clause:
        ident = _fc(pkg_clause, "package_identifier") or _fc(pkg_clause, "identifier")
        if ident:
            package = _t(src, ident)

    # ---- imports ----
    imports: List[GoImport] = []
    for imp_decl in _ad(root, "import_declaration"):
        for spec in _ad(imp_decl, "import_spec"):
            alias: Optional[str] = None
            path_lit: Optional[str] = None
            # порядок важен: alias (identifier|_) может стоять перед строковым литералом
            for ch in spec.children:
                ct = ch.type
                if ct == "identifier" and alias is None:
                    alias = _t(src, ch)
                elif ct in _STR_LITS and path_lit is None:
                    # убираем кавычки/бэктики
                    path_lit = _string_literal_text(src, ch)
                # небольшая оптимизация: выходим, когда уже нашли оба
                if alias is not None and path_lit is not None:
                    break
            if path_lit:
                imports.append(GoImport(path=path_lit, alias=alias))

    # ---- types ----
    types: List[GoType] = []
    for type_decl in _ad(root, "type_declaration"):
        for type_spec in _ad(type_decl, "type_spec"):
            tname_node = _fc(type_spec, "type_identifier")
            if not tname_node:
                continue
            tname = _t(src, tname_node)
            if not tname:
                continue

            line = type_spec.start_point[0] + 1
            struct_node = _fc(type_spec, "struct_type")
            if struct_node is not None:
                fld_list = _fc(struct_node, "field_declaration_list")
                fields = _parse_struct_fields(src, fld_list) if fld_list else []
                # сохраняем ровно то же, что было: list GoField (не меняем модель)
                types.append(GoType(name=tname, kind="struct", fields=fields or None, line=line))
                continue

            iface_node = _fc(type_spec, "interface_type")
            if iface_node is not None:
                methods: List[str] = []
                # method_spec может лежать на разной глубине — ищем потомков
                for ms in _ad(iface_node, "method_spec"):
                    methods.append(_t(src, ms).strip())
                types.append(GoType(name=tname, kind="interface", methods=methods or None, line=line))
                continue

            # alias / other: берём RHS как сырой текст (первый непустой кусок после имени)
            rhs: Optional[str] = None
            for ch in type_spec.children:
                if ch is tname_node:
                    continue
                txt = _t(src, ch).strip()
                if txt:
                    rhs = txt
                    break
            if rhs:
                types.append(GoType(name=tname, kind="alias", methods=[rhs], line=line))
            else:
                types.append(GoType(name=tname, kind="other", line=line))

    # ---- funcs & methods ----
    funcs: List[GoFunc] = []

    # Свободные функции
    for fn in _ad(root, "function_declaration"):
        fname_node = _fc(fn, "identifier")
        if not fname_node:
            continue
        fname = _t(src, fname_node)
        if not fname:
            continue

        in_pl = _fc(fn, "parameter_list")
        params = _parse_parameter_list(src, in_pl) if in_pl else []
        results = _parse_results(src, fn)
        doc = _collect_leading_comments(src, fn)
        line = fn.start_point[0] + 1

        funcs.append(GoFunc(
            name=fname,
            receiver=None,
            exported=(fname[0].isupper() if fname else False),
            params=params,
            results=results,
            doc=doc,
            line=line,
        ))

    # Методы (с ресивером)
    for md in _ad(root, "method_declaration"):
        # имя метода может быть field_identifier (обычно) или identifier (реже)
        mname_node = _fc(md, "field_identifier") or _fc(md, "identifier")
        if not mname_node:
            continue
        mname = _t(src, mname_node)
        if not mname:
            continue

        # ресивер — первый parameter_list
        recv: Optional[str] = None
        # вместо создания списка plists — найдём первый/второй на лету
        first_pl = None
        second_pl = None
        for ch in md.children:
            if ch.type == "parameter_list":
                if first_pl is None:
                    first_pl = ch
                elif second_pl is None:
                    second_pl = ch
                    break  # больше не нужно

        if first_pl is not None:
            # берём первый «осмысленный» тип из первой parameter_declaration
            for pd in _ad(first_pl, "parameter_declaration"):
                # первый ребёнок, который не identifier/comment — это и есть тип ресивера
                for ch in pd.children:
                    if ch.type not in ("identifier", "comment"):
                        rt = _t(src, ch).strip()
                        if rt:
                            recv = rt
                            break
                if recv:
                    break

        # параметры метода — второй parameter_list (если есть)
        params: List[GoParam] = _parse_parameter_list(src, second_pl) if second_pl is not None else []

        results = _parse_results(src, md)
        doc = _collect_leading_comments(src, md)
        line = md.start_point[0] + 1

        funcs.append(GoFunc(
            name=mname,
            receiver=recv,
            exported=(mname[0].isupper() if mname else False),
            params=params,
            results=results,
            doc=doc,
            line=line,
        ))

    return GoFileMeta(
        path=str(path),
        package=package,
        imports=imports,
        types=types,
        funcs=funcs,
    )

In [83]:
# ──────────────────────────────────────────────────────────────────────────────
# Публичный контракт (микро-оптимизация без смены логики)
# ──────────────────────────────────────────────────────────────────────────────
from typing import Dict, Iterable, Union
from pathlib import Path


def make_project_tree(PROJECT_PATH: Union[str, Path], IGNORE_DIRS: Iterable[str]) -> Dict:
    """
    Возвращает файловое дерево (dict) только с *.go, go.mod, go.sum.
    """
    root = Path(PROJECT_PATH).resolve()
    tree: Dict = {"name": root.name, "type": "dir", "children": {}}

    children = tree["children"]  # локальная ссылка

    def insert(path: Path):
        rel = path.relative_to(root)
        node = children
        parts = rel.parts
        for i, part in enumerate(parts):
            is_last = i == len(parts) - 1
            nd = node.get(part)
            if nd is None:
                nd = node[part] = {
                    "name": part,
                    "type": "file" if is_last else "dir",
                    "children": None if is_last else {},
                }
            if not is_last:
                node = nd["children"]

    # детерминированный порядок
    files = sorted(iter_go_files(root, IGNORE_DIRS), key=lambda p: (p.parent.as_posix(), p.name))
    for f in files:
        insert(f)

    return tree


def make_structures(PROJECT_PATH: Union[str, Path], IGNORE_DIRS: Iterable[str]) -> Dict[str, GoFileMeta]:
    """
    Возвращает {relative_path: GoFileMeta} по всем *.go с разбором пакетов/импортов/типов/функций.
    """
    root = Path(PROJECT_PATH).resolve()

    # используем ваш фабричный метод; если в ноутбуке уже есть make_go_parser() — применим его
    parser = make_go_parser()

    out: Dict[str, GoFileMeta] = {}
    # только .go файлы; детерминированный порядок
    files = sorted(iter_go_files(root, IGNORE_DIRS), key=lambda p: (p.parent.as_posix(), p.name))
    rel = Path.relative_to  # локальная ссылка на метод для микро-экономии лукапов

    for f in files:
        if f.suffix != ".go":
            continue
        meta = parse_go_file_structures(f, parser)
        out[str(rel(f, root))] = meta

    return out

In [84]:
# ──────────────────────────────────────────────────────────────────────────────
# Утилиты вывода (микро-оптимизация без изменения API/форматов)
# ──────────────────────────────────────────────────────────────────────────────

from dataclasses import asdict
from typing import Dict, List


def to_jsonable_structures(structs: Dict[str, GoFileMeta]) -> Dict[str, Dict]:
    # Локальные ссылки уменьшают оверхед на глобальные лукапы в больших словарях
    _asdict = asdict
    return {k: _asdict(v) for k, v in structs.items()}


def dump_structures_json(structs: Dict[str, GoFileMeta]) -> str:
    # Те же параметры, но минимизируем лишние лукапы
    return json.dumps(
        to_jsonable_structures(structs),
        ensure_ascii=False,
        indent=2,
    )


def tree_to_pretty_lines(tree: Dict, indent: str = "") -> List[str]:
    """
    Формирует список строк вида:
      root/
        dir1/
          file.go
        go.mod
    Предполагается, что каталог имеет "type": "dir", файл — "type": "file".
    """
    lines: List[str] = []
    _children = tree.get("children") or {}
    name = tree.get("name", "")
    node_type = tree.get("type", "dir")

    # Заголовок узла
    if node_type == "dir":
        lines.append(f"{indent}{name}/")
    else:
        # На случай если корень передали как file
        lines.append(f"{indent}{name}")

    # Дети (детерминированный порядок)
    for child_name in sorted(_children.keys()):
        node = _children[child_name]
        ntype = node.get("type", "file")
        if ntype == "dir":
            # Хвост рекурсивно
            lines.extend(tree_to_pretty_lines(node, indent + "  "))
        else:
            lines.append(f"{indent}  {child_name}")

    return lines


# proto анализ


In [85]:
# ──────────────────────────────────────────────────────────────────────────────
# Proto parser + workspace индексы + усиленный аннотатор (устойчивый резолв Reply)
# ──────────────────────────────────────────────────────────────────────────────
import re, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set
from dataclasses import dataclass, field

# ==== утилиты ====
def _get(obj: Any, attr: str, default=None):
    if isinstance(obj, dict):
        return obj.get(attr, default)
    return getattr(obj, attr, default)

def _iter(obj: Any, attr: str) -> List[Any]:
    v = _get(obj, attr, None)
    return list(v) if v else []

# нормализация go-типа до alias.Type
_GO_QTYPE = re.compile(r'^\s*(?:(?P<prefix>(?:\*|\[\s*\]|\.\.\.|map\s*\[[^\]]+\]\s*|chan\s*<?-?>?\s*)+)?)(?:(?P<alias>[A-Za-z_]\w*)\.)?(?P<type>[A-Za-z_]\w+)\s*$')
_CLEAN_WRAP = [
    (re.compile(r'^\s*\(\s*'), ''), (re.compile(r'\s*\)\s*$'), ''),           # внешние скобки
    (re.compile(r'^\s*\*+\s*'), ''),                                          # *T
    (re.compile(r'^\s*\[\s*\]\s*'), ''),                                      # []T
    (re.compile(r'^\s*map\s*\[[^\]]*\]\s*'), ''),                             # map[K]V
    (re.compile(r'^\s*chan\s*(?:<-|<-\s*)?'), ''),                            # chan T
    (re.compile(r'^\s*<-?\s*chan\s*'), ''),                                   # <-chan T
    (re.compile(r'^\s*\.{3}\s*'), ''),                                        # ...T
    (re.compile(r'<[^>]*>'), ''),                                             # generics <...>
]
def _strip_parens_all(s: str) -> str:
    s = s.strip()
    while s.startswith("(") and s.endswith(")"):
        depth = 0; ok = True
        for ch in s:
            if ch == "(": depth += 1
            elif ch == ")":
                depth -= 1
                if depth < 0: ok = False; break
        if ok and depth == 0:
            s = s[1:-1].strip()
        else:
            break
    return s

def _base_type_text(t: str) -> str:
    s = _strip_parens_all(t or "")
    for rx, repl in _CLEAN_WRAP:
        s = rx.sub(repl, s)
    return s.strip()

def _split_go_qual_type(t: str) -> Optional[Tuple[Optional[str], str]]:
    s = _base_type_text(t)
    m = _GO_QTYPE.match(s)
    return (m.group('alias'), m.group('type')) if m else None

# ==== Proto модели ====
@dataclass
class ProtoField:
    label: Optional[str]   # repeated|optional|required|None
    type: str              # scalar, message, map<...>, pkg.Type
    name: str
    number: Optional[int] = None
    options: Optional[str] = None
    line: Optional[int] = None
    doc: Optional[str] = None

@dataclass
class ProtoMessage:
    name: str              # полное имя для вложенных: Parent.Child
    fields: List[ProtoField] = field(default_factory=list)
    line: Optional[int] = None
    doc: Optional[str] = None

@dataclass
class ProtoFileMeta:
    path: str
    package: Optional[str] = None
    go_package: Optional[str] = None
    messages: List[ProtoMessage] = field(default_factory=list)
    options: Dict[str, Any] = field(default_factory=dict)

# ==== парсер .proto ====
_rx_package     = re.compile(r'^\s*package\s+([A-Za-z0-9_.]+)\s*;')
_rx_go_package  = re.compile(r'^\s*option\s+go_package\s*=\s*"([^"]+)"\s*;')
_rx_generic_opt = re.compile(r'^\s*option\s+([A-Za-z_]\w*)\s*=\s*(.+?)\s*;')
_rx_field = re.compile(
    r'^\s*'
    r'(?:(repeated|optional|required)\s+)?'
    r'(map\s*<[^>]+>|[A-Za-z_][\w\.]*)\s+'
    r'([A-Za-z_]\w*)\s*=\s*(\d+)'
    r'(?:\s*\[([^\]]+)\])?\s*;'
)

def parse_proto_file(path: Path) -> ProtoFileMeta:
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()

    def strip_inline_comment(s: str) -> str:
        s = re.sub(r'//.*', '', s)
        s = re.sub(r'/\*.*?\*/', '', s)
        return s

    def collect_leading_doc(i: int) -> Optional[str]:
        j = i - 1
        buf: List[str] = []
        in_block = False
        while j >= 0:
            raw = lines[j]
            if not in_block and raw.strip() == "": break
            if '*/' in raw and '/*' not in raw:
                in_block = True; buf.insert(0, raw); j -= 1; continue
            if in_block:
                buf.insert(0, raw)
                if '/*' in raw: in_block = False
                j -= 1; continue
            if raw.strip().startswith("//"):
                buf.insert(0, raw); j -= 1; continue
            break
        if not buf: return None
        cleaned: List[str] = []
        block_mode = False
        for t in buf:
            t = t.rstrip()
            if t.strip().startswith("/*"): block_mode = True
            if block_mode:
                t2 = re.sub(r'^\s*/\*+', '', t)
                t2 = re.sub(r'\*+/\s*$', '', t2)
                t2 = re.sub(r'^\s*\* ?', '', t2)
                cleaned.append(t2.strip())
                if '*/' in t: block_mode = False
            else:
                cleaned.append(re.sub(r'^\s*//\s?', '', t).strip())
        doc = "\n".join([c for c in cleaned if c.strip() != ""])
        return doc or None

    meta = ProtoFileMeta(path=str(path.resolve()))
    stack: List[Tuple[str, int, List[ProtoMessage]]] = []
    current_msgs: List[ProtoMessage] = []
    brace_depth = 0

    for i, raw in enumerate(lines):
        s_no_inline = strip_inline_comment(raw)
        s = s_no_inline.strip()

        # options / headers
        mopt = _rx_generic_opt.match(s)
        if mopt:
            k, v = mopt.groups()
            meta.options[k] = v.strip().strip('"')
        if meta.package is None:
            m = _rx_package.match(s)
            if m: meta.package = m.group(1)
        if meta.go_package is None:
            m = _rx_go_package.match(s)
            if m: meta.go_package = m.group(1)

        # message start
        if re.match(r'^\s*message\s+[A-Za-z_]\w*\s*\{', s):
            m2 = re.match(r'^\s*message\s+([A-Za-z_]\w*)', s)
            if m2:
                name = m2.group(1)
                doc = collect_leading_doc(i)
                msg = ProtoMessage(name=name, line=i+1, doc=doc, fields=[])
                if stack:
                    parent_prefix = stack[-1][0]
                    msg.name = f"{parent_prefix}.{name}"
                stack.append((msg.name, brace_depth, current_msgs))
                current_msgs = [msg]
                brace_depth += 1
                continue

        # учёт скобок
        open_count  = s.count("{")
        close_count = s.count("}")
        brace_depth += open_count

        # поля
        if current_msgs:
            mf = _rx_field.match(s)
            if mf:
                label, ftype, fname, num, opts = mf.groups()
                doc = collect_leading_doc(i)
                fld = ProtoField(
                    label=label, type=ftype.strip(), name=fname,
                    number=int(num), options=(opts.strip() if opts else None),
                    line=i+1, doc=doc
                )
                current_msgs[-1].fields.append(fld)

        brace_depth -= close_count
        while stack and brace_depth == stack[-1][1]:
            _, _, parent_acc = stack.pop()
            finished = current_msgs
            if stack:
                current_msgs = stack[-1][2]
            else:
                current_msgs = []
                meta.messages.extend(finished)

    if current_msgs:
        meta.messages.extend(current_msgs)

    return meta

# ==== Makefile → PROTOS_PATH ====
def read_protos_path_from_makefile(makefile_path: Path, project_root: Path) -> Optional[Path]:
    if not makefile_path.is_file():
        return None
    txt = makefile_path.read_text(encoding="utf-8", errors="ignore")
    # допускаем формы 'PROTOS_PATH = ./protocol' и 'PROTOS_PATH:=protocol'
    m = re.search(r'^\s*PROTOS_PATH\s*[:=]\s*(.+?)\s*$', txt, re.MULTILINE)
    if not m:
        return None
    val = m.group(1).strip().strip('"').strip("'")
    p = (project_root / val).resolve() if val.startswith(".") else Path(val).resolve()
    return p if p.exists() else None

# ==== workspace индексы ====
def _derive_aliases_for_proto_file(pf: ProtoFileMeta) -> Set[str]:
    aliases: Set[str] = set()
    go_pkg = pf.go_package or _get(pf.options, "go_package")
    if go_pkg:
        if ";" in go_pkg:
            path_part, alias = go_pkg.split(";", 1)
            alias = alias.strip()
            if alias: aliases.add(alias)
            tail = Path(path_part.strip()).name
            if tail: aliases.add(tail)
        else:
            tail = Path(go_pkg.strip()).name
            if tail: aliases.add(tail)
    if pf.package:
        aliases.add(pf.package.split(".")[-1])
    fdir = Path(pf.path).parent.name
    if fdir:
        aliases.add(fdir)
    return {a for a in aliases if a}

def parse_proto_workspace(proto_roots: List[Path]) -> List[ProtoFileMeta]:
    seen: Set[Path] = set()
    files: List[Path] = []
    for root in proto_roots:
        if not root or not root.exists():
            continue
        for p in root.rglob("*.proto"):
            rp = p.resolve()
            if rp not in seen:
                seen.add(rp); files.append(rp)
    protos: List[ProtoFileMeta] = []
    for p in files:
        try:
            protos.append(parse_proto_file(p))
        except Exception as e:
            print(f"[warn] proto parse failed: {p}: {e}")
    return protos

def build_workspace_message_index(proto_files: List[ProtoFileMeta]) -> Dict[str, Tuple[ProtoFileMeta, ProtoMessage]]:
    """
    Ключи:
      • alias.Short (alias из go_package; package; имя каталога)
      • package.Short
      • diralias.Short  (имя каталога .proto)
      • Short           (если уникально по короткому имени)
    """
    idx: Dict[str, Tuple[ProtoFileMeta, ProtoMessage]] = {}
    short_count: Dict[str, int] = {}

    # 0) посчитаем уникальность по коротким именам (Child у Parent.Child)
    for pf in proto_files:
        for m in pf.messages:
            short = m.name.split(".")[-1]
            short_count[short] = short_count.get(short, 0) + 1

    for pf in proto_files:
        pkg = (pf.package or "").split(".")[-1] if pf.package else None
        diralias = Path(pf.path).parent.name

        # derive aliases (go_package; pkg; dir)
        aliases = set()
        gp = pf.go_package or _get(pf.options, "go_package")
        if gp:
            if ";" in gp:
                path_part, alias = gp.split(";", 1)
                alias = alias.strip()
                if alias: aliases.add(alias)
                tail = Path(path_part.strip()).name
                if tail: aliases.add(tail)
            else:
                tail = Path(gp.strip()).name
                if tail: aliases.add(tail)
        if pkg:
            aliases.add(pkg)
        if diralias:
            aliases.add(diralias)

        # нормализуем к нижнему регистру для кейс-инсенситивного поиска alias
        aliases = {a.strip() for a in aliases if a}
        aliases_lc = {a.lower() for a in aliases}

        for m in pf.messages:
            short = m.name.split(".")[-1]
            # alias.Short
            for a in aliases:
                idx[f"{a}.{short}"] = (pf, m)
            # package.Short
            if pkg:
                idx[f"{pkg}.{short}"] = (pf, m)
            # diralias.Short (в нижнем регистре — добавляем как дополнительный ключ)
            idx[f"{diralias}.{short}"] = (pf, m)
            idx[f"{diralias.lower()}.{short}"] = (pf, m)
            for a in aliases_lc:
                idx[f"{a}.{short}"] = (pf, m)
            # Short — если уникально
            if short_count.get(short, 0) == 1:
                idx.setdefault(short, (pf, m))
    return idx

def build_workspace_alias_index(proto_files: List[ProtoFileMeta]) -> Dict[str, List[ProtoFileMeta]]:
    alias_to_pfs: Dict[str, List[ProtoFileMeta]] = {}
    for pf in proto_files:
        gp = pf.go_package or _get(pf.options, "go_package")
        aliases = set()
        if gp:
            if ";" in gp:
                path_part, alias = gp.split(";", 1)
                alias = alias.strip()
                if alias: aliases.add(alias)
                tail = Path(path_part.strip()).name
                if tail: aliases.add(tail)
            else:
                tail = Path(gp.strip()).name
                if tail: aliases.add(tail)
        if pf.package:
            aliases.add(pf.package.split(".")[-1])
        aliases.add(Path(pf.path).parent.name)

        # Нормализуем и добавляем дубликаты в нижнем регистре
        for a in list(aliases):
            alias_to_pfs.setdefault(a, []).append(pf)
            alias_to_pfs.setdefault(a.lower(), []).append(pf)
    return alias_to_pfs

# ==== аннотация функций с усиленным фоллбэком ====
def annotate_functions_with_io_types_enriched_workspace(
    structs_map: Dict[str, dict],
    include_paths: List[Path],
    path_map: Dict[str, Path],
    workspace_idx: Dict[str, Tuple[ProtoFileMeta, ProtoMessage]],
    workspace_alias_idx: Optional[Dict[str, List[ProtoFileMeta]]] = None,
    include_builtins: bool = False,   # если нужно отображать error/bool и пр.
) -> int:
    def _resolve_import_to_local(import_path: str,
                                 path_map: Dict[str, Path],
                                 include_paths: List[Path]) -> Optional[Path]:
        norm = import_path.strip("/").replace("\\", "/")
        best_prefix, best_root = "", None
        for prefix, root in path_map.items():
            pfx = prefix.strip("/").replace("\\", "/")
            if norm == pfx or norm.startswith(pfx + "/"):
                if len(pfx) > len(best_prefix):
                    best_prefix, best_root = pfx, root
        if best_root is not None:
            suffix = norm[len(best_prefix):].lstrip("/")
            cand = best_root.joinpath(suffix).with_suffix(".proto")
            if cand.is_file():
                return cand
        for base in include_paths:
            cand = base.joinpath(norm).with_suffix(".proto")
            if cand.is_file():
                return cand
        return None

    # импортированные .proto → alias
    alias_to_proto: Dict[str, ProtoFileMeta] = {}
    for _, meta in structs_map.items():
        for imp in meta.get("imports") or []:
            alias = imp.get("alias"); path = imp.get("path")
            if not alias or not path or alias in alias_to_proto:
                continue
            p = _resolve_import_to_local(path, path_map, include_paths)
            if p:
                try:
                    alias_to_proto[alias] = parse_proto_file(p)
                except Exception as e:
                    print(f"[warn] proto parse failed (import): {p}: {e}")

    # быстрый индекс alias.Short → (pf, msg) только из импортов
    alias_msg_idx: Dict[str, Tuple[ProtoFileMeta, ProtoMessage]] = {}
    for a, pf in alias_to_proto.items():
        for m in pf.messages:
            alias_msg_idx[f"{a}.{m.name.split('.')[-1]}"] = (pf, m)

    # локальные Go-структуры
    go_idx: Dict[str, Tuple[str, dict]] = {}
    for rel, meta in structs_map.items():
        for t in meta.get("types") or []:
            if t.get("kind") == "struct" and t.get("name"):
                go_idx.setdefault(t["name"], (rel, t))

    def _go_struct_definition(t: Dict[str, Any]) -> str:
        name = _get(t, "name", "Unknown")
        lines = [f"type {name} struct {{"]
        for f in _iter(t, "fields"):
            fname = _get(f, "name", ""); ftype = _get(f, "type", ""); tag = _get(f, "tag", None)
            if _get(f, "fields", None):
                lines.append(f"    {fname} struct {{ … }}")
            else:
                lines.append(f'    {fname} {ftype}' + (f' `{tag}`' if tag else ""))
        lines.append("}")
        return "\n".join(lines)

    def _proto_message_definition(m: ProtoMessage) -> str:
        mname = m.name.split(".")[-1]
        lines = [f"message {mname} {{"] 
        for pf in m.fields:
            head = f"  {pf.label} {pf.type} {pf.name}" if pf.label else f"  {pf.type} {pf.name}"
            tail = f" = {pf.number}" if pf.number is not None else ""
            lines.append(f"{head}{tail}" + (";" if not pf.options else f" [{pf.options}];"))
        lines.append("}")
        return "\n".join(lines)

    def _workspace_candidates_by_short(short_name: str) -> List[Tuple[ProtoFileMeta, ProtoMessage]]:
        cands: List[Tuple[ProtoFileMeta, ProtoMessage]] = []
        seen = set()
        for (pf, m) in workspace_idx.values():
            if m.name.split(".")[-1] == short_name and (id(pf), id(m)) not in seen:
                seen.add((id(pf), id(m)))
                cands.append((pf, m))
        return cands

    def _append_resolved(container: List[Dict[str, Any]], pf: ProtoFileMeta, m: ProtoMessage, fq_hint: Optional[str]=None):
        go_pkg = pf.go_package or _get(pf.options, "go_package")
        if fq_hint:
            fq = fq_hint
        else:
            if go_pkg and ";" in go_pkg:
                alias_best = go_pkg.split(";", 1)[1].strip()
            else:
                alias_best = (pf.package.split(".")[-1] if pf.package else Path(pf.path).parent.name)
            fq = f"{alias_best}.{m.name.split('.')[-1]}"
        container.append({
            "source": "proto", "name": m.name.split(".")[-1],
            "fq": fq, "file": pf.path,
            "line": m.line, "doc": m.doc,
            "definition": _proto_message_definition(m),
            "fields": [f.__dict__ for f in m.fields],
        })

    resolved = 0

    for _, meta in structs_map.items():
        for fn in _iter(meta, "funcs"):
            preferred_proto_files: Set[str] = set()

            # ----- параметры -----
            rparams: List[Dict[str, Any]] = []
            for p in _iter(fn, "params"):
                alias, base = _split_go_qual_type(_get(p, "type", "")) or (None, None)
                if not base:
                    continue

                # A) импортом
                key = f"{alias}.{base}" if alias else None
                if key and key in alias_msg_idx:
                    pf, m = alias_msg_idx[key]
                    preferred_proto_files.add(pf.path)
                    _append_resolved(rparams, pf, m, fq_hint=key); resolved += 1; continue

                # B) workspace: (1) по alias → файлы → message; (2) по ключу; (3) по короткому
                hit = None
                if not hit and alias and workspace_alias_idx:
                    for pf in workspace_alias_idx.get(alias, []):
                        for mm in pf.messages:
                            if mm.name.split(".")[-1] == base:
                                hit = (pf, mm); break
                        if hit: break
                if not hit:
                    hit = workspace_idx.get(f"{alias}.{base}") if alias else None
                if not hit:
                    cands = _workspace_candidates_by_short(base)
                    hit = cands[0] if cands else None
                if hit:
                    pf, m = hit
                    preferred_proto_files.add(pf.path)
                    _append_resolved(rparams, pf, m); resolved += 1; continue

                # C) локальная Go-структура
                g = go_idx.get(base)
                if g:
                    rel_path, t = g
                    rparams.append({
                        "source": "go", "name": _get(t, "name"),
                        "fq": _get(t, "name"), "file": rel_path,
                        "line": _get(t, "line", None), "doc": _get(t, "doc", None),
                        "definition": _go_struct_definition(t),
                        "fields": _iter(t, "fields"),
                    }); resolved += 1

            if rparams:
                fn["resolved_params"] = rparams

                        # ----- результаты -----
            rresults: List[Dict[str, Any]] = []
            for r in _get(fn, "results", []) or []:
                raw = str(r).strip()
                norm = _base_type_text(raw)  # снимет *, [], (), map, chan, <>
                alias, base = _split_go_qual_type(norm) or (None, None)
                if not base:
                    continue

                # (A) alias.Short по импортам
                key = f"{alias}.{base}" if alias else None
                if key and key in alias_msg_idx:
                    pf, m = alias_msg_idx[key]
                    _append_resolved(rresults, pf, m, fq_hint=key); resolved += 1; continue

                # (B1) теми же файлами, что у параметров
                hit = None
                if preferred_proto_files:
                    for (pf, mm) in _workspace_candidates_by_short(base):
                        if pf.path in preferred_proto_files:
                            hit = (pf, mm); break

                # (B2) workspace по alias → файлы → message (учитываем нижний регистр)
                if not hit and alias and workspace_alias_idx:
                    bucket = workspace_alias_idx.get(alias) or workspace_alias_idx.get(alias.lower())
                    if bucket:
                        for pf in bucket:
                            for mm in pf.messages:
                                if mm.name.split(".")[-1] == base:
                                    hit = (pf, mm); break
                            if hit: break

                # (B3) workspace по ключу (alias.Short) и в нижнем регистре
                if not hit and alias:
                    hit = workspace_idx.get(f"{alias}.{base}") or workspace_idx.get(f"{alias.lower()}.{base}")
                # (B4) workspace по короткому имени
                if not hit:
                    cands = _workspace_candidates_by_short(base)
                    hit = cands[0] if cands else None

                if hit:
                    pf, m = hit
                    _append_resolved(rresults, pf, m); resolved += 1; continue

                # (C) локальная Go-struct (на случай прямых структур)
                g = go_idx.get(base)
                if g:
                    rel_path, t = g
                    rresults.append({
                        "source": "go", "name": _get(t, "name"),
                        "fq": _get(t, "name"), "file": rel_path,
                        "line": _get(t, "line", None), "doc": _get(t, "doc", None),
                        "definition": _go_struct_definition(t),
                        "fields": _iter(t, "fields"),
                    }); resolved += 1

            if rresults:
                fn["resolved_results"] = rresults

    return resolved

# PlantUML диаграма

In [91]:
# ──────────────────────────────────────────────────────────────────────────────
# PLANTUML: вложенность с параметрами max_depth и merge_beyond
# ──────────────────────────────────────────────────────────────────────────────
import json, re
from typing import Any, Dict, List, Set, Tuple, Optional

# ====== базовые утилиты ======
def _is_json_str(x: Any) -> bool:
    return isinstance(x, str) and (x.lstrip().startswith("{") or x.lstrip().startswith("["))

def _get(obj: Any, attr: str, default=None):
    if isinstance(obj, dict):
        return obj.get(attr, default)
    return getattr(obj, attr, default)

def _iter(obj: Any, attr: str) -> List[Any]:
    v = _get(obj, attr, None)
    return list(v) if v else []

def _sanitize_ident(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z_]', '_', s)

def _short_typename(t: Any) -> str:
    s = str(t)
    s = re.sub(r'`[^`]*`', '', s)
    s = s.replace('...', '')
    s = re.sub(r'\bchan\b<?-?>?', ' ', s)
    s = re.sub(r'\bmap\s*$begin:math:display$[^$end:math:display$]*\]', ' ', s)
    s = re.sub(r'<[^>]*>', ' ', s)
    s = re.sub(r'[*$begin:math:display$$end:math:display$]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    parts = re.split(r'[/\.\s]', s)
    parts = [p for p in parts if p]
    return parts[-1] if parts else str(t).strip()

def _field_type_display(t: Any) -> str:
    s = str(t)
    s = re.sub(r'`[^`]*`', '', s).replace('"', '').replace('\n', ' ')
    return re.sub(r'\s+', ' ', s).strip()

def _index_structs_and_aliases(structs_map: Dict[str, Any]):
    struct_names: Set[str] = set()
    alias_rhs: Dict[str, str] = {}
    items: List[Tuple[str, str, Any]] = []
    name_counts: Dict[str, int] = {}

    for rel_path, meta in structs_map.items():
        pkg = _get(meta, "package", "") or ""
        for t in _iter(meta, "types"):
            kind = _get(t, "kind")
            name = _get(t, "name")
            if not name:
                continue
            if kind == "struct":
                items.append((rel_path, pkg, t))
                struct_names.add(name)
                name_counts[name] = name_counts.get(name, 0) + 1
            elif kind == "alias":
                rhs_list = _iter(t, "methods")
                rhs = rhs_list[0] if rhs_list else None
                if rhs:
                    alias_rhs[name] = str(rhs)
    return struct_names, alias_rhs, items, name_counts

def _resolve_to_struct_name(typename: str, struct_names: Set[str], alias_rhs: Dict[str, str], depth: int = 0) -> Optional[str]:
    if not typename:
        return None
    base = _short_typename(typename)
    if base in struct_names:
        return base
    if depth < 6 and base in alias_rhs:
        return _resolve_to_struct_name(alias_rhs[base], struct_names, alias_rhs, depth + 1)
    return None

def _fmt_params_for_sig(params: List[Any]) -> str:
    out = []
    for p in params or []:
        nm = _get(p, "name", None)
        tp = _short_typename(_get(p, "type", ""))
        out.append(f"{nm}:{tp}" if nm else tp)
    return ", ".join(out)

def _fmt_results_for_sig(results: List[Any]) -> str:
    if not results:
        return ""
    r = ", ".join(_short_typename(x) for x in results)
    return f" : {r}"

# ====== рендер вложенных контейнеров по ключам-путям ======
def _render_containers(containers_sorted: List[str], render_payload):
    """
    containers_sorted: список путей-контейнеров (e.g. "cmd/api", "cmd/api/handlers.go" или "pkg/service")
    render_payload(container_path:str, indent:str) -> List[str]
    """
    lines: List[str] = []
    open_stack: List[str] = []

    def _open_until(parts: List[str]):
        nonlocal open_stack
        i = 0
        while i < len(open_stack) and i < len(parts) and open_stack[i] == parts[i]:
            i += 1
        # close
        for _ in range(len(open_stack) - i):
            lines.append("}")
            open_stack.pop()
        # open
        for j in range(i, len(parts)):
            seg_path = "/".join(parts[:j+1])
            alias = _sanitize_ident(f"dir_{seg_path}")
            lines.append(f'package "{parts[j]}" as {alias} {{')
            open_stack.append(parts[j])

    for key in containers_sorted:
        parts = key.split("/") if key else []
        _open_until(parts[:-1])  # открыть до родителя
        leaf = parts[-1] if parts else "(root)"
        alias = _sanitize_ident(f"file_{key}")  # одинаково и для файла, и для агрегир. директории
        lines.append(f'package "{leaf}" as {alias} {{')
        lines.extend(render_payload(key, "  "))
        lines.append("}")

    # close remaining
    for _ in range(len(open_stack)):
        lines.append("}")
    return lines

def _container_key_for_file(rel: str, max_depth: int, merge_beyond: bool) -> str:
    """Возвращает ключ-контейнер для файла rel с учётом глубины/объединения."""
    parts = rel.split("/")
    dir_parts, file_name = parts[:-1], parts[-1]
    if max_depth < 0:
        max_depth = 0
    if len(dir_parts) <= max_depth or not merge_beyond:
        # контейнер = сам файл
        return rel
    # иначе контейнер = директория на уровне max_depth (объединение)
    return "/".join(dir_parts[:max_depth])  # без имени файла

# ──────────────────────────────────────────────────────────────────────────────
# 1) Component diagram
# ──────────────────────────────────────────────────────────────────────────────
def plantuml_components_allowmixing_filtered_grouped(
    data: Dict[str, Any] | str,
    max_depth: int = 99,
    merge_beyond: bool = False,
) -> str:
    if _is_json_str(data):
        data = json.loads(data)
    structs_map: Dict[str, Any] = data

    files_map = {k: v for k, v in structs_map.items() if not k.endswith("_test.go")}

    # индекс struct’ов
    struct_names, alias_rhs, struct_items, name_counts = _index_structs_and_aliases(files_map)

    # отображение
    class_alias_of_base: Dict[str, str] = {}
    struct_alias_display: Dict[str, str] = {}
    iface_alias_display: Dict[str, str] = {}
    func_alias_display: Dict[str, str] = {}
    struct_type_by_alias: Dict[str, Any] = {}
    func_meta_by_alias: Dict[str, Any] = {}

    # распределение по контейнерам
    structs_by_container: Dict[str, List[str]] = {}
    ifaces_by_container: Dict[str, List[str]] = {}
    funcs_by_container: Dict[str, List[str]] = {}

    # structs
    for rel_path, pkg, t in struct_items:
        name = _get(t, "name")
        disp = name if name_counts.get(name, 0) == 1 else (f"{pkg}.{name}" if pkg else name)
        a = _sanitize_ident(f"struct_{disp}")
        struct_alias_display[a] = disp
        struct_type_by_alias[a] = t
        class_alias_of_base.setdefault(name, a)
        cont = _container_key_for_file(rel_path, max_depth, merge_beyond)
        structs_by_container.setdefault(cont, []).append(a)

    # interfaces
    for rel, meta in files_map.items():
        pkg = _get(meta, "package", "") or ""
        for t in _iter(meta, "types"):
            if _get(t, "kind") != "interface":
                continue
            iname = _get(t, "name")
            if not iname:
                continue
            disp = iname if name_counts.get(iname, 0) == 1 else (f"{pkg}.{iname}" if pkg else iname)
            a = _sanitize_ident(f"iface_{disp}")
            iface_alias_display[a] = disp
            cont = _container_key_for_file(rel, max_depth, merge_beyond)
            ifaces_by_container.setdefault(cont, []).append(a)

    # functions/methods → component (без init)
    for rel, meta in files_map.items():
        for fn in _iter(meta, "funcs"):
            fname = _get(fn, "name", "")
            if fname == "init":
                continue
            recv  = _get(fn, "receiver", None)
            in_sig  = _fmt_params_for_sig(_iter(fn, "params"))
            out_sig = _fmt_results_for_sig(_get(fn, "results", []) or [])
            disp = f"({ _short_typename(recv)}) {fname}({in_sig}){out_sig}" if recv else f"{fname}({in_sig}){out_sig}"
            a = _sanitize_ident(f"fn_{rel}__{disp}")
            func_alias_display[a] = disp.replace('"', "'")
            func_meta_by_alias[a] = {
                "receiver": recv,
                "params": _iter(fn, "params"),
                "results": _get(fn, "results", []) or [],
            }
            cont = _container_key_for_file(rel, max_depth, merge_beyond)
            funcs_by_container.setdefault(cont, []).append(a)

    # связи struct↔struct
    edges: List[str] = []
    for s_alias, t in struct_type_by_alias.items():
        for fld in _iter(t, "fields"):
            tgt = _resolve_to_struct_name(_get(fld, "type", ""), struct_names, alias_rhs)
            if not tgt:
                continue
            ta = class_alias_of_base.get(tgt)
            if not ta or ta == s_alias:
                continue
            if bool(_get(fld, "embedded", False)):
                edges.append(f"  {s_alias} *-- {ta}")
            else:
                edges.append(f"  {s_alias} o-- {ta}")

    # функции ↔ структуры (и фильтр пустых функций)
    non_empty_funcs: Set[str] = set()
    for f_alias, meta in func_meta_by_alias.items():
        for p in meta["params"]:
            tgt = _resolve_to_struct_name(_get(p, "type", ""), struct_names, alias_rhs)
            if tgt:
                ta = class_alias_of_base.get(tgt)
                if ta:
                    edges.append(f"  {f_alias} ..> {ta} : in")
                    non_empty_funcs.add(f_alias)
        for r in meta["results"]:
            tgt = _resolve_to_struct_name(r, struct_names, alias_rhs)
            if tgt:
                ta = class_alias_of_base.get(tgt)
                if ta:
                    edges.append(f"  {f_alias} ..> {ta} : out")
                    non_empty_funcs.add(f_alias)
        recv = meta.get("receiver")
        if recv:
            base = _short_typename(recv)
            ta = class_alias_of_base.get(base)
            if ta:
                edges.append(f"  {f_alias} -- {ta} : receiver")
                non_empty_funcs.add(f_alias)

    # рендер контейнеров
    containers = sorted(set(
        list(structs_by_container.keys()) +
        list(ifaces_by_container.keys()) +
        list(funcs_by_container.keys())
    ))
    def _render_container_payload(key: str, indent: str) -> List[str]:
        out: List[str] = []
        for a in sorted(structs_by_container.get(key, [])):
            out.append(f'{indent}struct "{struct_alias_display[a]}" as {a}')
        for a in sorted(ifaces_by_container.get(key, [])):
            out.append(f'{indent}interface "{iface_alias_display[a]}" as {a}')
        for a in sorted([x for x in funcs_by_container.get(key, []) if x in non_empty_funcs]):
            out.append(f'{indent}component "{func_alias_display[a]}" as {a}')
        return out

    lines: List[str] = ["@startuml", "allowmixing", "hide empty members"]
    lines.extend(_render_containers(containers, _render_container_payload))

    # оставить рёбра только к реально выведенным узлам
    rendered_nodes: Set[str] = set()
    for key in containers:
        rendered_nodes.update(structs_by_container.get(key, []))
        rendered_nodes.update(ifaces_by_container.get(key, []))
        rendered_nodes.update([a for a in funcs_by_container.get(key, []) if a in non_empty_funcs])

    for e in edges:
        parts = e.strip().split()
        if len(parts) >= 3 and parts[0] in rendered_nodes and parts[2] in rendered_nodes:
            lines.append(e)

    lines += [
        "legend left",
        "  struct — Go struct; interface — Go interface; component — функция/метод",
        f"  Вложенность ограничена max_depth={max_depth}; merge_beyond={merge_beyond}",
        "endlegend",
        "@enduml",
    ]
    return "\n".join(lines)

# ──────────────────────────────────────────────────────────────────────────────
# 2) Class/Struct diagram
# ──────────────────────────────────────────────────────────────────────────────
def plantuml_classes_structs_filtered_grouped(
    data: Dict[str, Any] | str,
    max_depth: int = 99,
    merge_beyond: bool = False,
) -> str:
    if _is_json_str(data):
        data = json.loads(data)
    files_map: Dict[str, Any] = {k: v for k, v in data.items() if not k.endswith("_test.go")}

    struct_names, alias_rhs, items, name_counts = _index_structs_and_aliases(files_map)

    class_alias_display: Dict[str, str] = {}
    type_by_alias: Dict[str, Any] = {}
    base_to_alias: Dict[str, str] = {}
    classes_by_container: Dict[str, List[str]] = {}

    for rel, pkg, t in items:
        name = _get(t, "name")
        disp = name if name_counts.get(name, 0) == 1 else (f"{pkg}.{name}" if pkg else name)
        a = _sanitize_ident(f"struct_{disp}")
        class_alias_display[a] = disp
        type_by_alias[a] = t
        base_to_alias.setdefault(name, a)
        cont = _container_key_for_file(rel, max_depth, merge_beyond)
        classes_by_container.setdefault(cont, []).append(a)

    containers = sorted(classes_by_container.keys())
    def _render_container_payload(key: str, indent: str) -> List[str]:
        out: List[str] = []
        for a in sorted(classes_by_container.get(key, [])):
            out.append(f'{indent}class "{class_alias_display[a]}" as {a}')
            t = type_by_alias[a]
            for fld in _iter(t, "fields"):
                fname = _get(fld, "name", "")
                ftype = _field_type_display(_get(fld, "type", ""))
                sub = _get(fld, "fields", None)
                if sub:
                    out.append(f'{indent}  {a} : +{fname} : struct{{…}}')
                else:
                    out.append(f'{indent}  {a} : +{fname} : {ftype}')
        return out

    lines: List[str] = ["@startuml", "hide empty members"]
    lines.extend(_render_containers(containers, _render_container_payload))

    # связи class↔class по полям
    for a, t in type_by_alias.items():
        for fld in _iter(t, "fields"):
            tgt = _resolve_to_struct_name(_get(fld, "type", ""), struct_names, alias_rhs)
            if not tgt: continue
            ta = base_to_alias.get(tgt)
            if not ta or ta == a: continue
            if bool(_get(fld, "embedded", False)):
                lines.append(f"  {a} *-- {ta}")
            else:
                lines.append(f"  {a} o-- {ta}")

    lines += [
        "legend left",
        f"  Вложенность ограничена max_depth={max_depth}; merge_beyond={merge_beyond}",
        "endlegend",
        "@enduml",
    ]
    return "\n".join(lines)

# ──────────────────────────────────────────────────────────────────────────────
# 3) Interface реализация
# ──────────────────────────────────────────────────────────────────────────────
_NAME_BEFORE_PAREN = re.compile(r'^\s*([A-Za-z_]\w*)\s*\(')
def _iface_required_method_names(t_iface: Any) -> Set[str]:
    req: Set[str] = set()
    for raw in _iter(t_iface, "methods") or []:
        m = _NAME_BEFORE_PAREN.match(str(raw))
        if m: 
            req.add(m.group(1))
    return req

def plantuml_interfaces_implements_grouped(
    data: Dict[str, Any] | str,
    max_depth: int = 99,
    merge_beyond: bool = False,
) -> str:
    if _is_json_str(data):
        data = json.loads(data)
    files_map: Dict[str, Any] = {k: v for k, v in data.items() if not k.endswith("_test.go")}

    struct_names, alias_rhs, struct_items, name_counts = _index_structs_and_aliases(files_map)

    struct_alias_by_container: Dict[str, List[str]] = {}
    struct_alias_display: Dict[str, str] = {}
    base_to_alias: Dict[str, str] = {}

    for rel, pkg, t in struct_items:
        nm = _get(t, "name")
        disp = nm if name_counts.get(nm, 0) <= 1 else (f"{pkg}.{nm}" if pkg else nm)
        a = _sanitize_ident(f"struct_{disp}")
        struct_alias_display[a] = disp
        base_to_alias.setdefault(nm, a)
        cont = _container_key_for_file(rel, max_depth, merge_beyond)
        struct_alias_by_container.setdefault(cont, []).append(a)

    iface_alias_by_container: Dict[str, List[str]] = {}
    iface_alias_display: Dict[str, str] = {}
    iface_type_by_alias: Dict[str, Any] = {}
    iface_name_counts: Dict[str, int] = {}
    for meta in files_map.values():
        for t in _iter(meta, "types"):
            if _get(t, "kind") == "interface":
                n = _get(t, "name")
                if n: 
                    iface_name_counts[n] = iface_name_counts.get(n, 0) + 1
    for rel, meta in files_map.items():
        pkg = _get(meta, "package", "") or ""
        for t in _iter(meta, "types"):
            if _get(t, "kind") != "interface": continue
            nm = _get(t, "name") 
            if not nm: 
                continue
            disp = nm if iface_name_counts.get(nm, 0) == 1 else (f"{pkg}.{nm}" if pkg else nm)
            a = _sanitize_ident(f"iface_{disp}")
            iface_alias_display[a] = disp
            iface_type_by_alias[a] = t
            cont = _container_key_for_file(rel, max_depth, merge_beyond)
            iface_alias_by_container.setdefault(cont, []).append(a)

    # методы структур (по ресиверам)
    struct_methods: Dict[str, Set[str]] = {}
    for rel, meta in files_map.items():
        for fn in _iter(meta, "funcs"):
            recv = _get(fn, "receiver", None)
            if not recv: continue
            base = _short_typename(recv)
            mname = _get(fn, "name", "")
            if not mname: 
                continue
            struct_methods.setdefault(base, set()).add(mname)

    # рёбра реализаций
    impl_edges: List[str] = []
    for a_iface, t_iface in iface_type_by_alias.items():
        req = _iface_required_method_names(t_iface)
        if not req: continue
        for base_struct, have in struct_methods.items():
            if req.issubset(have):
                a_struct = base_to_alias.get(base_struct)
                if a_struct:
                    impl_edges.append(f"  {a_struct} ..|> {a_iface}")

    # рендер
    containers = sorted(set(list(struct_alias_by_container.keys()) + list(iface_alias_by_container.keys())))
    def _render_container_payload(key: str, indent: str) -> List[str]:
        out: List[str] = []
        for a in sorted(struct_alias_by_container.get(key, [])):
            out.append(f'{indent}struct "{struct_alias_display[a]}" as {a}')
        for a in sorted(iface_alias_by_container.get(key, [])):
            out.append(f'{indent}interface "{iface_alias_display[a]}" as {a}')
        return out

    lines: List[str] = ["@startuml", "hide empty members"]
    lines.extend(_render_containers(containers, _render_container_payload))
    lines.extend(sorted(set(impl_edges)))
    lines += [
        "legend left",
        f"  Вложенность ограничена max_depth={max_depth}; merge_beyond={merge_beyond}",
        "endlegend",
        "@enduml",
    ]
    return "\n".join(lines)

# ──────────────────────────────────────────────────────────────────────────────
# 4) Dependencies (file/package) с вложенностью и агрегацией
# ──────────────────────────────────────────────────────────────────────────────
def plantuml_dependencies_grouped(
    data: Dict[str, Any] | str,
    aggregate: str = "file",
    max_depth: int = 99,
    merge_beyond: bool = False,
) -> str:
    if _is_json_str(data):
        data = json.loads(data)
    files_map: Dict[str, Any] = {k: v for k, v in data.items() if not k.endswith("_test.go")}

    lines: List[str] = ["@startuml", "hide empty members"]

    if aggregate == "file":
        # file → imports
        file_imports: Dict[str, Set[str]] = {}
        for rel, meta in files_map.items():
            imps = set()
            for imp in _iter(meta, "imports"):
                p = _get(imp, "path", "")
                if p: imps.add(p)
            file_imports[rel] = imps

        # источники — контейнеры по max_depth/merge_beyond
        containers = sorted({ _container_key_for_file(rel, max_depth, merge_beyond) for rel in file_imports.keys() })

        # собрать, какие файлы попали в какой контейнер
        files_in_container: Dict[str, List[str]] = {}
        for rel in file_imports.keys():
            key = _container_key_for_file(rel, max_depth, merge_beyond)
            files_in_container.setdefault(key, []).append(rel)

        def _render_container_payload(key: str, indent: str) -> List[str]:
            # рисуем заглушки-файлы только если не merge (иначе считаем, что компоненты/типы будут в других диаграммах)
            out: List[str] = []
            # ничего внутри — контейнер только как рамка
            return out

        lines.extend(_render_containers(containers, _render_container_payload))

        # сгенерим дерево импорта (как вложенные компоненты по их пути)
        def imp_alias(path: str) -> str:
            return _sanitize_ident(f"imp_{path}")

        all_imps: Set[str] = set().union(*file_imports.values()) if file_imports else set()
        imp_containers = sorted(all_imps)
        def _render_imp_payload(key: str, indent: str) -> List[str]:
            alias = imp_alias(key)
            return [f'{indent}component "{key.split("/")[-1]}" as {alias}']
        # рендер импортов
        lines.extend(_render_containers([p + ".import" for p in imp_containers],
                                        lambda rel, indent: _render_imp_payload(rel[:-7], indent)))

        # рёбра: файл-контейнер ..> импорт
        for key, files in files_in_container.items():
            src_alias = _sanitize_ident(f"file_{key}")
            for rel in files:
                for p in file_imports[rel]:
                    dst = _sanitize_ident(f"imp_{p}")
                    lines.append(f"  {src_alias} ..> {dst}")

    else:
        # aggregate=package: узлы — пакеты; импорты — сгруппированные пути
        pkg_to_imports: Dict[str, Set[str]] = {}
        for rel, meta in files_map.items():
            pkg = _get(meta, "package", "") or "(no-pkg)"
            s = pkg_to_imports.setdefault(pkg, set())
            for imp in _iter(meta, "imports"):
                p = _get(imp, "path", "")
                if p: s.add(p)

        # пакеты как плоские узлы (без вложенности — это уже уровень Go-пакета)
        for pkg in sorted(pkg_to_imports.keys()):
            alias = _sanitize_ident(f"pkg_{pkg}")
            lines.append(f'package "{pkg}" as {alias}')

        # импорты как вложенные контейнеры по пути
        all_imps: Set[str] = set().union(*pkg_to_imports.values()) if pkg_to_imports else set()
        def _render_imp_payload(key: str, indent: str) -> List[str]:
            alias = _sanitize_ident(f"imp_{key}")
            return [f'{indent}component "{key.split("/")[-1]}" as {alias}']
        lines.extend(_render_containers([p + ".import" for p in sorted(all_imps)],
                                        lambda rel, indent: _render_imp_payload(rel[:-7], indent)))

        for pkg, imps in pkg_to_imports.items():
            src = _sanitize_ident(f"pkg_{pkg}")
            for p in imps:
                dst = _sanitize_ident(f"imp_{p}")
                lines.append(f"  {src} ..> {dst}")

    lines += [
        "legend left",
        f"  Вложенность ограничена max_depth={max_depth}; merge_beyond={merge_beyond}",
        "endlegend",
        "@enduml",
    ]
    return "\n".join(lines)

# Полный пайплайн: Go → JSON → расширение через .proto → PlantUML

In [145]:
# ──────────────────────────────────────────────────────────────────────────────
# FULL PIPELINE (упорядочено, без двойной аннотации, с Makefile PROTOS_PATH)
# ──────────────────────────────────────────────────────────────────────────────
import json, re, time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set

# =========[ 0. КОНФИГ ]=========
PROJECT_PATH = Path("/home/jovyan/work/tree_docs/example/k8sgpt/")   # корень Go-проекта
OUT_DIR      = Path("/home/jovyan/work/tree_docs/example/")               # куда писать артефакты

MAKEFILE_PATH      = PROJECT_PATH / "Makefile"                             # Makefile с PROTOS_PATH
PROTO_FALLBACK_DIR = PROJECT_PATH / "protocol"                             # fallback, если PROTOS_PATH не найден

# Внешние импорты (если нужны)
PROTO_PATH_MAP: Dict[str, Path] = {
    # "gitlab.com/kuber/proto": Path("/home/jovyan/work/third_party/kuber/proto"),
    # "google/protobuf": Path("/home/jovyan/work/third_party/google/protobuf"),
}
# Игноры
IGNORE_DIRS = {'.git','vendor','node_modules','bin','dist','out','build','.idea','.vscode'}

# Настройки диаграмм
DIAG_MAX_DEPTH    = 2
DIAG_MERGE_BEYOND = True

# Включать диагностический вывод по нерешённым типам?
DEBUG_UNRESOLVED = True

t0 = time.time()

# =========[ 1. Go → JSON ]=========
structs = make_structures(PROJECT_PATH, IGNORE_DIRS=IGNORE_DIRS)
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_JSON = OUT_DIR / "result.json"
with open(RESULT_JSON, "w", encoding="utf-8") as f:
    json.dump(json.loads(dump_structures_json(structs)), f, ensure_ascii=False, indent=2)

t1 = time.time()

# =========[ 2. Опционально: разворачиваем proto-поля внутри Go-структур ]=========
with open(RESULT_JSON, "r", encoding="utf-8") as f:
    go_structs = json.load(f)

expanded_count = 0
if 'expand_proto_fields_in_structs' in globals():
    try:
        # Важно: передаём include_paths позже, когда узнаем proto_dir; здесь оставим как есть
        expanded_count = expand_proto_fields_in_structs(go_structs, [PROJECT_PATH, PROTO_FALLBACK_DIR], PROTO_PATH_MAP)
    except Exception as e:
        print(f"[warn] expand_proto_fields_in_structs failed: {e}")

RESULT_EXP = OUT_DIR / "result_expanded.json"
with open(RESULT_EXP, "w", encoding="utf-8") as f:
    json.dump(go_structs, f, ensure_ascii=False, indent=2)

t2 = time.time()

# =========[ 3. Workspace .proto индекс ]=========
# 3.1 Получаем PROTOS_PATH из Makefile (или fallback)
proto_dir = read_protos_path_from_makefile(MAKEFILE_PATH, PROJECT_PATH) or PROTO_FALLBACK_DIR
if not proto_dir.exists():
    print(f"[warn] PROTOS_PATH not found, fallback missing: {proto_dir}")

# 3.2 Строим индексы по воркспейсу
proto_roots = [proto_dir]
workspace_protos = parse_proto_workspace(proto_roots)

workspace_idx     = build_workspace_message_index(workspace_protos)
workspace_alias_idx = build_workspace_alias_index(workspace_protos)

# Включим найденный корень в include-пути
PROTO_INCLUDE_PATHS: List[Path] = [PROJECT_PATH, PROTO_FALLBACK_DIR]
if proto_dir not in PROTO_INCLUDE_PATHS:
    PROTO_INCLUDE_PATHS.append(proto_dir)

t3 = time.time()

# =========[ 4. Аннотация функций (единственный вызов) ]=========
before_snapshot = json.loads(json.dumps(go_structs))  # для диагностики

resolved_cnt = annotate_functions_with_io_types_enriched_workspace(
    go_structs,
    include_paths=PROTO_INCLUDE_PATHS,
    path_map=PROTO_PATH_MAP,
    workspace_idx=workspace_idx,
    workspace_alias_idx=workspace_alias_idx,
)

# Диагностика (опционально): покажем, что не удалось резолвить
if DEBUG_UNRESOLVED:
    def _short(s): 
        return str(s).strip()
    for rel, meta in go_structs.items():
        for fn in meta.get("funcs") or []:
            # входы
            want_params = [ _short(p.get("type","")) for p in (fn.get("params") or []) ]
            have_params = [ (rp.get("source"), rp.get("name")) for rp in fn.get("resolved_params", []) ]
            if want_params and not have_params:
                print(f"[debug] unresolved params in {rel}:{fn.get('name')} → {want_params}")
            # выходы
            want_results = [ _short(r) for r in (fn.get("results") or []) ]
            # фильтруем error/bool и т.п. если их не резолвим как сообщения
            want_proto_like = [x for x in want_results if "error" not in x]
            have_results = [ (rr.get("source"), rr.get("name")) for rr in fn.get("resolved_results", []) ]
            if want_proto_like and not have_results:
                print(f"[debug] unresolved results in {rel}:{fn.get('name')} → {want_results}")

RESULT_IO = OUT_DIR / "result_with_io.json"
with open(RESULT_IO, "w", encoding="utf-8") as f:
    json.dump(go_structs, f, ensure_ascii=False, indent=2)

t4 = time.time()

# =========[ 5. Диаграммы (если генераторы доступны) ]=========
PUMLS = []
if 'plantuml_components_allowmixing_filtered_grouped' in globals():
    PUMLS.append(("component_grouped.puml",
                  plantuml_components_allowmixing_filtered_grouped(go_structs, max_depth=DIAG_MAX_DEPTH, merge_beyond=DIAG_MERGE_BEYOND)))
if 'plantuml_classes_structs_filtered_grouped' in globals():
    PUMLS.append(("classes_grouped.puml",
                  plantuml_classes_structs_filtered_grouped(go_structs, max_depth=DIAG_MAX_DEPTH, merge_beyond=DIAG_MERGE_BEYOND)))
if 'plantuml_interfaces_implements_grouped' in globals():
    PUMLS.append(("implements_grouped.puml",
                  plantuml_interfaces_implements_grouped(go_structs, max_depth=DIAG_MAX_DEPTH, merge_beyond=DIAG_MERGE_BEYOND)))
if 'plantuml_dependencies_grouped' in globals():
    PUMLS.append(("deps_file_grouped.puml",
                  plantuml_dependencies_grouped(go_structs, aggregate="file", max_depth=DIAG_MAX_DEPTH, merge_beyond=DIAG_MERGE_BEYOND)))
    PUMLS.append(("deps_pkg_grouped.puml",
                  plantuml_dependencies_grouped(go_structs, aggregate="package", max_depth=DIAG_MAX_DEPTH, merge_beyond=DIAG_MERGE_BEYOND)))

for name, content in PUMLS:
    (OUT_DIR / name).write_text(content, encoding="utf-8")

t5 = time.time()

# =========[ 6. Отчёт ]=========
files_total   = len(structs)
types_total   = sum(len(m.types or []) for m in structs.values())
funcs_total   = sum(len(m.funcs or []) for m in structs.values())
imports_total = sum(len(m.imports or []) for m in structs.values())

print("— Пайплайн завершён —")
print(f"Go-файлов обработано     : {files_total}")
print(f"Импортов найдено         : {imports_total}")
print(f"Типов (struct/iface)     : {types_total}")
print(f"Функций/методов          : {funcs_total}")
print(f"Развёрнуто proto-полей   : {expanded_count}")
print(f"Разрешено I/O типов      : {resolved_cnt}")
print(f"Proto файлов (workspace) : {len(workspace_protos)}")
print(f"Proto messages (workspace): {sum(len(p.messages) for p in workspace_protos)}")
print()
print(f"JSON (Go)                : {RESULT_JSON}")
print(f"JSON (Go+proto expanded) : {RESULT_EXP}")
print(f"JSON (с I/O аннотациями) : {RESULT_IO}")
for name, _ in PUMLS:
    print(f"PUML                     : {OUT_DIR / name}")
print()
print(f"Время: parse {t1-t0:.3f}с | expand {t2-t1:.3f}с | protoIdx {t3-t2:.3f}с | annotate {t4-t3:.3f}с | diagrams {t5-t4:.3f}с | total {t5-t0:.3f}с")

[warn] PROTOS_PATH not found, fallback missing: /home/jovyan/work/tree_docs/example/k8sgpt/protocol
[debug] unresolved results in cmd/root.go:getLegacyConfigFilePath → ['string', 'error']
[debug] unresolved results in cmd/root.go:getConfigFilePath → ['string']
[debug] unresolved params in cmd/root.go:Execute → ['string', 'string', 'string']
[debug] unresolved params in cmd/root_test.go:TestInitConfig_VerboseFlag → ['*testing.T']
[debug] unresolved params in cmd/generate/generate.go:printInstructions → ['bool', 'string']
[debug] unresolved params in cmd/generate/generate.go:openbrowser → ['string']
[debug] unresolved params in pkg/ai/amazonbedrock.go:validateInferenceProfileArn → ['string']
[debug] unresolved results in pkg/ai/amazonbedrock.go:validateInferenceProfileArn → ['bool']
[debug] unresolved params in pkg/ai/amazonbedrock.go:validateModelArn → ['string']
[debug] unresolved results in pkg/ai/amazonbedrock.go:validateModelArn → ['bool']
[debug] unresolved params in pkg/ai/amazonb

# README

In [146]:
from pathlib import Path
from datetime import datetime
from jinja2 import Environment, FileSystemLoader, StrictUndefined

TEMPLATES_DIR = Path("/home/jovyan/work/tree_docs/templates")      # ваш каталог с .j2 файлами
OUT_ROOT = PROJECT_PATH            # куда писать md

env = Environment(
    loader=FileSystemLoader(str(TEMPLATES_DIR)),
    trim_blocks=True, 
    lstrip_blocks=True,
    undefined=StrictUndefined,
)

def render(tpl_name: str, ctx: dict) -> str:
    return env.get_template(tpl_name).render(**ctx)

# ---- подготовка контекста для PROJECT_DOCS.md ----
project_tree = [...]  # список строк дерева .go (как раньше собирали)
dirs_ctx = [{"dir": drel, "readme_rel": str((PROJECT_PATH / ("" if drel=="." else drel) / "README.md").relative_to(PROJECT_PATH)),
             "readme_name": "README.md"} for drel in sorted(dirs_with_go.keys())]

ctx_docs = {
    "project_name": PROJECT_PATH.name,
    "project_tree": project_tree,
    "dirs": dirs_ctx,
    "test_link": "TESTS.md",
    "description_include": "",
    "contacts_include": "",
    "generated_at": datetime.now().isoformat(timespec="seconds"),
}
(OUT_ROOT / "PROJECT_DOCS.md").write_text(render("project_docs.md.j2", ctx_docs), encoding="utf-8")

# ──────────────────────────────────────────────────────────────────────────────
# ТЕСТЫ: сбор данных и рендер через tests.md.j2 (только Test*)
# ──────────────────────────────────────────────────────────────────────────────
import json

# Загрузка данных
go_map = json.loads(RESULT_JSON.read_text(encoding="utf-8"))

def _is_test_file(rel: str) -> bool:
    return rel.endswith("_test.go")

def _dir_of(rel: str) -> str:
    return str(Path(rel).parent).replace("\\", "/") or "."

def _collect_tests(fn_list):
    """Возвращает список только Test* с подготовленной сигнатурой и метаданными."""
    tests = []
    for f in fn_list or []:
        nm = f.get("name", "")
        if not nm.startswith("Test"):
            continue
        # сигнатура: имя(параметры) -> результаты
        in_params = []
        for p in f.get("params") or []:
            nm_p = p.get("name"); tp = p.get("type")
            in_params.append(f"{nm_p}: {tp}" if nm_p else str(tp))
        sig_out = ", ".join(str(r) for r in (f.get("results") or []))
        signature = f"{nm}({', '.join(in_params)})" + (f" -> {sig_out}" if sig_out else "")
        tests.append({
            "name": nm,
            "line": f.get("line"),
            "doc": (f.get("doc") or "").strip() or None,
            "signature": signature,
        })
    return tests

# Агрегация по директориям
dirs_raw = {}  # dir -> {"files": int, "tests": int, "files_list":[ ... ]}
totals = {"files": 0, "tests": 0}

for rel, meta in go_map.items():
    if not _is_test_file(rel):
        continue
    tests = _collect_tests(meta.get("funcs"))
    d = _dir_of(rel)

    bucket = dirs_raw.setdefault(d, {"files": 0, "tests": 0, "files_list": []})
    bucket["files"] += 1
    bucket["tests"] += len(tests)
    totals["files"] += 1
    totals["tests"] += len(tests)

    bucket["files_list"].append({
        "rel": rel,
        "pkg": meta.get("package"),
        "tests": tests,  # [{name,line,doc,signature}]
    })

# Упорядочим директории и файлы
dirs_ordered = []
for d, agg in sorted(dirs_raw.items(), key=lambda kv: kv[0]):
    files_list = sorted(agg["files_list"], key=lambda x: x["rel"])
    dirs_ordered.append({
        "dir": d,
        "files": agg["files"],
        "tests": agg["tests"],
        "files_list": files_list,
    })

# Рендер через шаблон
ctx_tests = {
    "totals": totals,
    "dirs_ordered": dirs_ordered,
    "generated_at": datetime.now().isoformat(timespec="seconds"),
}
(OUT_ROOT / "TESTS.md").write_text(render("tests.md.j2", ctx_tests), encoding="utf-8")

# ---- контекст для dir_readme.md ----
def build_dir_readme_ctx(dir_rel: str) -> dict:
    files = []
    for rel, meta in go_structs.items():
        if rel.endswith("_test.go"): continue
        if (dir_rel == "." and "/" not in rel) or rel.startswith(dir_rel + "/") or (dir_rel != "." and rel == dir_rel):
            funcs = []
            for fn in meta.get("funcs") or []:
                # подпишем сигнатуру
                in_params = []
                for p in fn.get("params") or []:
                    nm = p.get("name"); tp = p.get("type")
                    in_params.append(f"{nm}: {tp}" if nm else str(tp))
                out_rs = ", ".join(str(r) for r in (fn.get("results") or []))
                sig = f"{fn.get('name')}({', '.join(in_params)})" + (f" -> {out_rs}" if out_rs else "")
                funcs.append({
                    "sig": sig,
                    "line": fn.get("line"),
                    "doc": fn.get("doc"),
                    "resolved_params": fn.get("resolved_params") or [],
                    "resolved_results": fn.get("resolved_results") or [],
                })
            if funcs:
                files.append({"rel": rel, "funcs": funcs})
    return {
        "project_name": PROJECT_PATH.name,
        "dir_rel": dir_rel,
        "files": files,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    }

for drel in sorted(dirs_with_go.keys()):
    ctx = build_dir_readme_ctx(drel)
    out = render("dir_readme.md.j2", ctx)
    (OUT_ROOT / ("" if drel=="." else drel) / "README.md").write_text(out, encoding="utf-8")

FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/work/tree_docs/example/k8sgpt/server/README.md'